# Notebook 03 — 3D Patch Extraction

This notebook extracts fixed-size 3D CT patches and corresponding targets from the **persistent preprocessed RibFrac dataset** produced by Notebook 02.

## Development strategy
Only **RibFrac128** is processed initially. Full training/validation extraction remains disabled until the complete end-to-end pipeline is validated.

## Methodological principles
- Reuse the patient-level split created in Notebook 02; do not recreate it here.
- Keep all patches from a patient within that patient's assigned split.
- Use identical coordinates for CT and label patches.
- CT patch shape: `(64, 64, 64, 1)`.
- Segmentation target: binary background/fracture.
- Classification target: five classes: background, displaced, non-displaced, buckle, segmental.
- Map RibFrac annotation IDs to fracture classes using `ribfrac-train-info-1.csv`.
- Exclude ambiguous annotations (`label_code = -1`) from targets.
- Control background sampling with a negative-to-positive ratio.
- Do not perform a random patch-level train/validation split.

In [ ]:
# Cell 1 — Import libraries

import gc
import random
import numpy as np
import pandas as pd

from pathlib import Path
from collections import Counter

print("Libraries imported successfully.")

In [ ]:
# Cell 2 — Reproducibility

SEED = 42

np.random.seed(SEED)
random.seed(SEED)

print("Random seed:", SEED)

In [ ]:
# Cell 3 — Define persistent input and patch-output directories

DATASET_ROOT = Path("/kaggle/input/ribfrac-preprocessed-dev")

PREPROCESSED_ROOT = DATASET_ROOT / "ribfrac_preprocessed"

PREPROCESSED_IMAGE_DIR = PREPROCESSED_ROOT / "images"
PREPROCESSED_LABEL_DIR = PREPROCESSED_ROOT / "labels"
PREPROCESSED_METADATA_DIR = PREPROCESSED_ROOT / "metadata"

# RibFrac annotation metadata is included at the dataset root.
INFO_PATH = (
    DATASET_ROOT
    / "ribfrac_metadata"
    / "ribfrac-train-info-1.csv"
)

# Patch files are temporary until the complete pipeline is validated.
BASE_DIR = Path("/kaggle/working")
PATCH_DIR = BASE_DIR / "patch_dataset"
PATCH_MANIFEST_DIR = PATCH_DIR / "manifests"

PATCH_DIR.mkdir(parents=True, exist_ok=True)
PATCH_MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

print("Persistent dataset:", DATASET_ROOT)
print("Preprocessed images:", PREPROCESSED_IMAGE_DIR)
print("Preprocessed labels:", PREPROCESSED_LABEL_DIR)
print("Preprocessed metadata:", PREPROCESSED_METADATA_DIR)
print("RibFrac annotation metadata:", INFO_PATH)
print("Patch output:", PATCH_DIR)

In [ ]:
# Cell 4 — Verify persistent inputs

SPLIT_PATH = PREPROCESSED_METADATA_DIR / "patient_split.csv"

for path, description in [
    (PREPROCESSED_IMAGE_DIR, "preprocessed image directory"),
    (PREPROCESSED_LABEL_DIR, "preprocessed label directory"),
    (SPLIT_PATH, "patient split file"),
    (INFO_PATH, "RibFrac annotation metadata"),
]:
    if not path.exists():
        raise FileNotFoundError(
            f"{description} not found: {path}"
        )

print("All required persistent inputs found.")
print("Patient split:", SPLIT_PATH)
print("Annotation metadata:", INFO_PATH)

In [ ]:
# Cell 5 — Load patient-level split and annotation metadata

split_df = pd.read_csv(SPLIT_PATH)
info_df = pd.read_csv(INFO_PATH)

print("Patient split shape:", split_df.shape)
print("Split columns:", split_df.columns.tolist())

print("\nAnnotation metadata shape:", info_df.shape)
print("Annotation columns:", info_df.columns.tolist())

display(split_df.head())
display(info_df.head())

In [ ]:
# Cell 6 — Validate and load the fixed patient-level split

required_split_columns = {"public_id", "split"}

if not required_split_columns.issubset(split_df.columns):
    raise ValueError(
        "patient_split.csv must contain 'public_id' and 'split' columns."
    )

split_df["public_id"] = split_df["public_id"].astype(str)
split_df["split"] = split_df["split"].astype(str)

allowed_splits = {"train", "validation", "test"}

if not set(split_df["split"]).issubset(allowed_splits):
    raise ValueError(
        f"Unexpected split values: {sorted(set(split_df['split']))}"
    )

if split_df["public_id"].duplicated().any():
    raise ValueError(
        "Duplicate patient IDs found in patient_split.csv."
    )

train_patients = split_df.loc[
    split_df["split"] == "train", "public_id"
].tolist()

val_patients = split_df.loc[
    split_df["split"] == "validation", "public_id"
].tolist()

test_patients = split_df.loc[
    split_df["split"] == "test", "public_id"
].tolist()

print("Training patients:", len(train_patients))
print("Validation patients:", len(val_patients))
print("Test patients:", len(test_patients))

assert not (set(train_patients) & set(val_patients))
assert not (set(train_patients) & set(test_patients))
assert not (set(val_patients) & set(test_patients))

print("Patient-level split integrity: PASS")

In [ ]:
# Cell 7 — Build annotation lookup

required_info_columns = {"public_id", "label_id", "label_code"}

if not required_info_columns.issubset(info_df.columns):
    raise ValueError(
        "Annotation metadata must contain "
        "'public_id', 'label_id', and 'label_code'."
    )

info_df["public_id"] = info_df["public_id"].astype(str)
info_df["label_id"] = info_df["label_id"].astype(int)
info_df["label_code"] = info_df["label_code"].astype(int)

annotation_lookup = {}

for _, row in info_df.iterrows():
    patient_id = row["public_id"]
    label_id = int(row["label_id"])
    label_code = int(row["label_code"])
    annotation_lookup[(patient_id, label_id)] = label_code

print("Annotation lookup entries:", len(annotation_lookup))
print(
    "Observed label codes:",
    sorted(info_df["label_code"].unique().tolist())
)
print("Expected fracture codes: [1, 2, 3, 4]")

In [ ]:
# Cell 8 — Patch extraction parameters

PATCH_SIZE = (64, 64, 64)
STRIDE = (32, 32, 32)

# Approximately one background patch for each fracture-positive patch.
NEGATIVE_TO_POSITIVE_RATIO = 1.0

MIN_FRACTURE_VOXELS = 1

# Ambiguous annotations (label_code = -1) are excluded.
IGNORE_AMBIGUOUS = True

CLASS_NAMES = {
    0: "background",
    1: "displaced",
    2: "non-displaced",
    3: "buckle",
    4: "segmental"
}

print("Patch size:", PATCH_SIZE)
print("Stride:", STRIDE)
print("Negative/positive ratio:", NEGATIVE_TO_POSITIVE_RATIO)
print("Minimum fracture voxels:", MIN_FRACTURE_VOXELS)
print("Classification classes:", CLASS_NAMES)

In [ ]:
# Cell 9 — Patch utility functions

def generate_patch_origins(
    volume_shape,
    patch_size=PATCH_SIZE,
    stride=STRIDE
):
    # Generate origins with complete volume coverage.

    max_x = volume_shape[0] - patch_size[0]
    max_y = volume_shape[1] - patch_size[1]
    max_z = volume_shape[2] - patch_size[2]

    if min(max_x, max_y, max_z) < 0:
        raise ValueError(
            f"Volume {volume_shape} is smaller than patch size {patch_size}."
        )

    xs = list(range(0, max_x + 1, stride[0]))
    ys = list(range(0, max_y + 1, stride[1]))
    zs = list(range(0, max_z + 1, stride[2]))

    if xs[-1] != max_x:
        xs.append(max_x)

    if ys[-1] != max_y:
        ys.append(max_y)

    if zs[-1] != max_z:
        zs.append(max_z)

    return [
        (x, y, z)
        for x in xs
        for y in ys
        for z in zs
    ]


def extract_patch(
    volume,
    origin,
    patch_size=PATCH_SIZE
):
    x, y, z = origin
    px, py, pz = patch_size

    return volume[
        x:x + px,
        y:y + py,
        z:z + pz
    ]


print("Patch utility functions defined.")

In [ ]:
# Cell 10 — Determine valid annotation IDs for one patient

def get_patient_annotation_maps(patient_id):

    patient_rows = info_df.loc[
        info_df["public_id"] == str(patient_id)
    ]

    if patient_rows.empty:
        raise ValueError(
            f"No annotation metadata found for {patient_id}."
        )

    valid_ids = {
        int(row["label_id"])
        for _, row in patient_rows.iterrows()
        if int(row["label_code"]) in (1, 2, 3, 4)
    }

    ambiguous_ids = {
        int(row["label_id"])
        for _, row in patient_rows.iterrows()
        if int(row["label_code"]) == -1
    }

    return valid_ids, ambiguous_ids


print("Patient annotation-map function defined.")

In [ ]:
# Cell 11 — Determine patch targets

def get_patch_targets(
    patient_id,
    label_patch,
    annotation_lookup,
    ignore_ambiguous=True
):
    # 0 = background; 1-4 = fracture classes.

    unique_ids, counts = np.unique(
        label_patch,
        return_counts=True
    )

    class_counts = Counter()
    valid_fracture_voxels = 0
    ambiguous_voxels = 0

    segmentation_mask = np.zeros(
        label_patch.shape,
        dtype=np.float32
    )

    for label_id, count in zip(unique_ids, counts):

        label_id = int(label_id)
        count = int(count)

        if label_id == 0:
            continue

        label_code = annotation_lookup.get(
            (str(patient_id), label_id),
            None
        )

        if label_code is None:
            continue

        if label_code == -1:
            ambiguous_voxels += count
            continue

        if label_code in (1, 2, 3, 4):
            class_counts[int(label_code)] += count
            valid_fracture_voxels += count

            segmentation_mask[
                label_patch == label_id
            ] = 1.0

    if not class_counts:
        patch_class = 0
    else:
        # If multiple fracture classes occur, use the class
        # occupying the greatest number of valid voxels.
        patch_class = max(
            class_counts,
            key=class_counts.get
        )

    return (
        int(patch_class),
        segmentation_mask,
        int(valid_fracture_voxels),
        int(ambiguous_voxels)
    )


print("Patch target function defined.")

In [ ]:
# Cell 12 — Preprocessed patient paths

def get_preprocessed_paths(patient_id):

    ct_path = (
        PREPROCESSED_IMAGE_DIR
        / f"{patient_id}-image.npy"
    )

    label_path = (
        PREPROCESSED_LABEL_DIR
        / f"{patient_id}-label.npy"
    )

    return ct_path, label_path


def check_patient_files(patient_id):

    ct_path, label_path = get_preprocessed_paths(
        patient_id
    )

    return ct_path.exists() and label_path.exists()


print("Patient path functions defined.")

In [ ]:
# Cell 13 — Verify persistent RibFrac128 input

TEST_PATIENT_ID = "RibFrac128"

ct_path, label_path = get_preprocessed_paths(
    TEST_PATIENT_ID
)

print("CT:", ct_path)
print("Label:", label_path)
print("CT exists:", ct_path.exists())
print("Label exists:", label_path.exists())

if not ct_path.exists():
    raise FileNotFoundError(
        f"RibFrac128 CT not found: {ct_path}"
    )

if not label_path.exists():
    raise FileNotFoundError(
        f"RibFrac128 label not found: {label_path}"
    )

ct = np.load(ct_path, mmap_mode="r")
label = np.load(label_path, mmap_mode="r")

print("CT shape:", ct.shape)
print("Label shape:", label.shape)
print("CT dtype:", ct.dtype)
print("Label dtype:", label.dtype)

assert ct.shape == label.shape
assert ct.ndim == 3
assert label.ndim == 3

print("Input validation: PASS")

In [ ]:
# Cell 14 — Test RibFrac128 annotation mapping

valid_ids, ambiguous_ids = get_patient_annotation_maps(
    TEST_PATIENT_ID
)

print(
    "Valid fracture annotation IDs:",
    sorted(valid_ids)
)

print(
    "Ambiguous annotation IDs:",
    sorted(ambiguous_ids)
)

if not valid_ids:
    raise ValueError(
        "No valid fracture annotation IDs were found "
        f"for {TEST_PATIENT_ID}."
    )

print("Annotation mapping validation: PASS")

In [ ]:
# Cell 15 — Extract patches from one patient

def extract_patient_patches(patient_id):

    ct_path, label_path = get_preprocessed_paths(
        patient_id
    )

    if not ct_path.exists():
        raise FileNotFoundError(
            f"CT file not found: {ct_path}"
        )

    if not label_path.exists():
        raise FileNotFoundError(
            f"Label file not found: {label_path}"
        )

    ct = np.load(ct_path, mmap_mode="r")
    label = np.load(label_path, mmap_mode="r")

    if ct.shape != label.shape:
        raise ValueError(
            f"CT/label shape mismatch for {patient_id}: "
            f"{ct.shape} vs {label.shape}"
        )

    origins = generate_patch_origins(ct.shape)

    positive_records = []
    negative_records = []
    ignored_records = []

    # First pass: identify candidate patch locations.
    for origin in origins:

        label_patch = extract_patch(label, origin)

        (
            patch_class,
            segmentation_mask,
            fracture_voxels,
            ambiguous_voxels
        ) = get_patch_targets(
            patient_id,
            label_patch,
            annotation_lookup,
            IGNORE_AMBIGUOUS
        )

        record = {
            "patient_id": str(patient_id),
            "origin_x": int(origin[0]),
            "origin_y": int(origin[1]),
            "origin_z": int(origin[2]),
            "class": int(patch_class),
            "fracture_voxels": int(fracture_voxels),
            "ambiguous_voxels": int(ambiguous_voxels)
        }

        # Ambiguous-only patches are ignored.
        if fracture_voxels == 0 and ambiguous_voxels > 0:
            ignored_records.append(record)
            continue

        if patch_class == 0:
            negative_records.append(record)

        elif fracture_voxels >= MIN_FRACTURE_VOXELS:
            positive_records.append(record)

    # Background sampling.
    rng = np.random.default_rng(SEED)

    max_negative = int(
        len(positive_records)
        * NEGATIVE_TO_POSITIVE_RATIO
    )

    if len(negative_records) > max_negative:

        indices = rng.choice(
            len(negative_records),
            size=max_negative,
            replace=False
        )

        negative_records = [
            negative_records[i]
            for i in indices
        ]

    records = positive_records + negative_records
    rng.shuffle(records)

    # Extract selected patches.
    X_patches = []
    Y_seg_patches = []
    Y_class = []

    for record in records:

        origin = (
            record["origin_x"],
            record["origin_y"],
            record["origin_z"]
        )

        ct_patch = extract_patch(ct, origin)
        label_patch = extract_patch(label, origin)

        (
            patch_class,
            segmentation_mask,
            fracture_voxels,
            ambiguous_voxels
        ) = get_patch_targets(
            patient_id,
            label_patch,
            annotation_lookup,
            IGNORE_AMBIGUOUS
        )

        X_patches.append(
            ct_patch[..., np.newaxis].astype(np.float32)
        )

        Y_seg_patches.append(
            segmentation_mask[..., np.newaxis].astype(np.float32)
        )

        Y_class.append(int(patch_class))

    X_patches = np.asarray(
        X_patches,
        dtype=np.float32
    )

    Y_seg_patches = np.asarray(
        Y_seg_patches,
        dtype=np.float32
    )

    Y_class = np.asarray(
        Y_class,
        dtype=np.int64
    )

    manifest = pd.DataFrame(records)

    print(f"Patient: {patient_id}")
    print("Candidate patch locations:", len(origins))
    print("Positive patches:", len(positive_records))
    print("Selected negative patches:", len(negative_records))
    print(
        "Ignored ambiguous-only patches:",
        len(ignored_records)
    )
    print("Final patches:", len(records))

    return (
        X_patches,
        Y_seg_patches,
        Y_class,
        manifest
    )


print("Patient patch-extraction function defined.")

In [ ]:
# Cell 16 — Test patch extraction with RibFrac128

(
    X_patient,
    Y_seg_patient,
    Y_class_patient,
    manifest_patient
) = extract_patient_patches(
    TEST_PATIENT_ID
)

print("\nX shape:", X_patient.shape)
print("Y_seg shape:", Y_seg_patient.shape)
print("Y_class shape:", Y_class_patient.shape)

print("\nClass distribution:")

class_distribution = (
    pd.Series(Y_class_patient)
    .value_counts()
    .sort_index()
)

display(
    class_distribution
    .rename_axis("class")
    .reset_index(name="patches")
)

display(manifest_patient.head())

In [ ]:
# Cell 17 — Validate RibFrac128 extracted patches

assert X_patient.ndim == 5
assert Y_seg_patient.ndim == 5
assert Y_class_patient.ndim == 1

assert X_patient.shape[1:] == (64, 64, 64, 1)
assert Y_seg_patient.shape[1:] == (64, 64, 64, 1)

assert X_patient.shape[0] == Y_seg_patient.shape[0]
assert X_patient.shape[0] == Y_class_patient.shape[0]

assert np.isfinite(X_patient).all()
assert np.isfinite(Y_seg_patient).all()

assert set(np.unique(Y_seg_patient)).issubset({0.0, 1.0})
assert set(np.unique(Y_class_patient)).issubset({0, 1, 2, 3, 4})

assert X_patient.shape[0] > 0

print("Patch validation: PASS")
print("Input patch shape:", X_patient.shape[1:])
print("Segmentation target shape:", Y_seg_patient.shape[1:])
print(
    "Classification classes present:",
    sorted(np.unique(Y_class_patient).tolist())
)

In [ ]:
# Cell 18 — Save RibFrac128 patch test

TEST_PATCH_DIR = PATCH_DIR / TEST_PATIENT_ID

TEST_PATCH_DIR.mkdir(
    parents=True,
    exist_ok=True
)

np.save(TEST_PATCH_DIR / "X.npy", X_patient)
np.save(TEST_PATCH_DIR / "Y_seg.npy", Y_seg_patient)
np.save(TEST_PATCH_DIR / "Y_class.npy", Y_class_patient)

manifest_patient.to_csv(
    PATCH_MANIFEST_DIR / f"{TEST_PATIENT_ID}_manifest.csv",
    index=False
)

print("RibFrac128 patches saved.")
print("Output directory:", TEST_PATCH_DIR)

In [ ]:
# Cell 19 — Reload saved RibFrac128 patches

saved_X = np.load(
    TEST_PATCH_DIR / "X.npy",
    mmap_mode="r"
)

saved_Y_seg = np.load(
    TEST_PATCH_DIR / "Y_seg.npy",
    mmap_mode="r"
)

saved_Y_class = np.load(
    TEST_PATCH_DIR / "Y_class.npy",
    mmap_mode="r"
)

saved_manifest = pd.read_csv(
    PATCH_MANIFEST_DIR / f"{TEST_PATIENT_ID}_manifest.csv"
)

assert saved_X.shape == X_patient.shape
assert saved_Y_seg.shape == Y_seg_patient.shape
assert saved_Y_class.shape == Y_class_patient.shape
assert len(saved_manifest) == X_patient.shape[0]

print("Saved patch reload validation: PASS")
print("Reloaded X:", saved_X.shape)
print("Reloaded Y_seg:", saved_Y_seg.shape)
print("Reloaded Y_class:", saved_Y_class.shape)
print("Reloaded manifest rows:", len(saved_manifest))

In [ ]:
# Cell 20 — Release RibFrac128 test arrays

del X_patient
del Y_seg_patient
del Y_class_patient
del manifest_patient

del saved_X
del saved_Y_seg
del saved_Y_class
del saved_manifest

del ct
del label

gc.collect()

print("Temporary RibFrac128 arrays released.")

## Full split-wise extraction

The full dataset extraction is deliberately disabled during pipeline development.

After Notebooks 03–07 have been validated with RibFrac128:

- Set `PROCESS_TRAINING_SET = True` to process the 192 development training patients.
- Set `PROCESS_VALIDATION_SET = True` to process the 48 development validation patients.
- Keep `PROCESS_TEST_SET = False` for the current development workflow unless ground-truth labels are available for the selected test cases.

**Important:** RibFrac128 is currently an end-to-end development case. Its results must not be reported as thesis model-performance results.

In [ ]:
# Cell 21 — Extraction switches

PROCESS_TRAINING_SET = False
PROCESS_VALIDATION_SET = False
PROCESS_TEST_SET = False

print("Training:", PROCESS_TRAINING_SET)
print("Validation:", PROCESS_VALIDATION_SET)
print("Test:", PROCESS_TEST_SET)

In [ ]:
# Cell 22 — Process one patient split

def process_patient_split(
    patient_list,
    split_name
):

    split_dir = PATCH_DIR / split_name
    split_dir.mkdir(parents=True, exist_ok=True)

    manifest_dir = PATCH_MANIFEST_DIR / split_name
    manifest_dir.mkdir(parents=True, exist_ok=True)

    combined_manifests = []
    processed_patients = []
    skipped_patients = []

    for i, patient_id in enumerate(
        patient_list,
        start=1
    ):

        print(
            f"\n[{split_name}] "
            f"Patient {i}/{len(patient_list)}: {patient_id}"
        )

        if not check_patient_files(patient_id):
            print("Preprocessed files not found — skipping.")
            skipped_patients.append(patient_id)
            continue

        (
            X_p,
            Y_s,
            Y_c,
            manifest
        ) = extract_patient_patches(patient_id)

        patient_dir = split_dir / patient_id
        patient_dir.mkdir(parents=True, exist_ok=True)

        np.save(patient_dir / "X.npy", X_p)
        np.save(patient_dir / "Y_seg.npy", Y_s)
        np.save(patient_dir / "Y_class.npy", Y_c)

        manifest = manifest.copy()
        manifest["split"] = split_name

        manifest.to_csv(
            manifest_dir / f"{patient_id}.csv",
            index=False
        )

        combined_manifests.append(manifest)
        processed_patients.append(patient_id)

        del X_p
        del Y_s
        del Y_c
        del manifest

        gc.collect()

    if combined_manifests:

        combined_manifest = pd.concat(
            combined_manifests,
            ignore_index=True
        )

        combined_manifest.to_csv(
            PATCH_MANIFEST_DIR / f"{split_name}_manifest.csv",
            index=False
        )

    else:
        combined_manifest = pd.DataFrame()

    summary = pd.DataFrame({
        "split": [split_name],
        "patients_requested": [len(patient_list)],
        "patients_processed": [len(processed_patients)],
        "patients_skipped": [len(skipped_patients)],
        "patches": [len(combined_manifest)]
    })

    summary.to_csv(
        PATCH_MANIFEST_DIR / f"{split_name}_summary.csv",
        index=False
    )

    print(f"\n{split_name} complete.")
    print("Patients processed:", len(processed_patients))
    print("Patients skipped:", len(skipped_patients))
    print("Total patches:", len(combined_manifest))

    return (
        combined_manifest,
        processed_patients,
        skipped_patients
    )


print("Split-processing function defined.")

In [ ]:
# Cell 23 — Process training patients

if PROCESS_TRAINING_SET:

    (
        train_manifest,
        train_processed,
        train_skipped
    ) = process_patient_split(
        train_patients,
        "train"
    )

else:

    print("Training extraction disabled.")

In [ ]:
# Cell 24 — Process validation patients

if PROCESS_VALIDATION_SET:

    (
        validation_manifest,
        validation_processed,
        validation_skipped
    ) = process_patient_split(
        val_patients,
        "validation"
    )

else:

    print("Validation extraction disabled.")

In [ ]:
# Cell 25 — Process test patients

if PROCESS_TEST_SET:

    (
        test_manifest,
        test_processed,
        test_skipped
    ) = process_patient_split(
        test_patients,
        "test"
    )

else:

    print("Test extraction disabled.")
    print(
        "Public RibFrac test labels are not available "
        "for the official test set."
    )

In [ ]:
# Cell 26 — Final patch dataset summary

print("Patch dataset:", PATCH_DIR)

print("\nSaved NumPy files:")

saved_npy_files = list(PATCH_DIR.rglob("*.npy"))

for path in sorted(saved_npy_files):

    size_mb = path.stat().st_size / (1024 ** 2)

    print(
        f"{path} | {size_mb:.2f} MB"
    )

print("\nSaved manifest files:")

for path in sorted(PATCH_MANIFEST_DIR.rglob("*.csv")):
    print(path)

print("\nTotal NumPy files:", len(saved_npy_files))

## Notebook 03 completion criteria

Notebook 03 is ready for Notebook 04 when the RibFrac128 test produces:

1. A valid persistent CT input.
2. A valid persistent label input.
3. Patient-level split integrity verification.
4. CT patches with shape `(64, 64, 64, 1)`.
5. Binary segmentation targets with values `{0, 1}`.
6. Five-class classification labels in `{0, 1, 2, 3, 4}`.
7. Ambiguous-only patches excluded.
8. CT and target patches aligned spatially.
9. Saved patch files can be reloaded successfully.
10. No patch-level train/validation split is performed.

### Target definitions

| Target | Meaning |
|---|---|
| Segmentation `0` | Background |
| Segmentation `1` | Fracture |
| Classification `0` | Background |
| Classification `1` | Displaced |
| Classification `2` | Non-displaced |
| Classification `3` | Buckle |
| Classification `4` | Segmental |

**Important:** Notebook 04 must therefore use a **five-class softmax classification head**, rather than the earlier single-sigmoid binary classification head.